# tinyLMTune — Text Summarization

This notebook demonstrates 3 ways to train TinyBERT for **summarization** using tinyLMTune:

1. **Synthetic data** — auto-generated via Flan-T5/Mistral
2. **Benchmark data** — real HuggingFace dataset (xsum)
3. **Raw user data** — your own text, structured or unstructured

Each example runs the full pipeline: data → token analysis → search space recommendation → GA optimisation → model save → inference.

## Setup

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

# Install if needed (uncomment):
!pip install -e ../../tinylmtune_v2/

In [1]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

# Install if needed (uncomment):
# !pip install -e .

from tinylmtune import optimize_slm, TinyInference, print_token_analysis, print_recommendation, plot_results

2026-05-25 07:36:01,677 | datasets | PyTorch version 2.4.1.post300 available.
2026-05-25 07:36:01,679 | datasets | TensorFlow version 2.17.0 available.
2026-05-25 07:36:03.106434: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-25 07:36:03.119944: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-25 07:36:03.124632: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-25 07:36:03.134838: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following in

---
## Example 1 — Synthetic Data (via Flan-T5)

No data needed. Flan-T5/Mistral generates training data from a topic prompt.

**Requirements:** Flan-T5 must be installed and running (`pip install sentencepiece`), with `mistral` model pulled (``).

In [2]:
best = optimize_slm(
    task="summarization",
    corpus_prompt="Generate news articles with summaries about technology and science",
    n_examples=1000,
    pop_size=4,
    generations=2,
    max_len=192,
    output_dir="models/summarization_synthetic",
)
print("Best config:", best)

2026-05-25 07:36:14,607 | tinylmtune._internal.pipeline | Model will be saved to: /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/summarization_synthetic
2026-05-25 07:36:14,608 | tinylmtune._internal.pipeline | No user data — generating synthetic corpus
2026-05-25 07:36:14,608 | tinylmtune._internal.corpus_gen | Generating 1000 summarization examples about 'Generate news articles with summaries about technology and science' using google/flan-t5-small ...
2026-05-25 07:36:14,609 | tinylmtune._internal.llm_backend | Loading LLM: google/flan-t5-small ...
2026-05-25 07:36:16,014 | tinylmtune._internal.llm_backend | LLM loaded on cuda (76961152 params)
2026-05-25 07:36:52,743 | tinylmtune._internal.corpus_gen | Generated 50 / 1000 examples (48 valid so far)
2026-05-25 07:37:25,852 | tinylmtune._internal.corpus_gen | Generated 100 / 1000 examples (95 valid so far)
2026-05-25 07:37:59,356 | tinylmtune._internal.corpus_gen | Generated 150 / 1000 examples (142 valid so far)
2026-05-25 

Epoch,Training Loss,Validation Loss
1,6.359900,3.165722
2,2.262700,1.817137
3,1.395200,1.685731
4,1.150400,1.702298
5,1.059400,1.738669


There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


2026-05-25 07:47:38,372 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 1.6857309341430664, 'eval_runtime': 0.1406, 'eval_samples_per_second': 1358.017, 'eval_steps_per_second': 85.32, 'epoch': 5.0}
2026-05-25 07:47:38,605 | tinylmtune._internal.trainer | Training: lr=0.0004805092517635441 bs=4 epochs=4 dropout=0.16 attn_drop=0.07 grad_accum=8 scheduler=constant_with_warmup label_smooth=0.031 grad_norm=1.9


Epoch,Training Loss,Validation Loss
0,6.131300,3.154093
1,2.324700,1.646184
2,1.217800,1.219430
3,0.792900,1.090415


...

2026-05-25 07:49:11,335 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 1.3611009120941162, 'eval_runtime': 0.2394, 'eval_samples_per_second': 797.904, 'eval_steps_per_second': 200.52, 'epoch': 3.9633507853403143}
2026-05-25 07:49:11,336 | tinylmtune._internal.ga_optimizer | Gen 2/2 — best=0.4822  avg=0.4528  worst=0.4235
2026-05-25 07:49:11,337 | tinylmtune._internal.pipeline | Best config (fitness=0.4822): {'learning_rate': 0.0004805092517635441, 'batch_size': 4, 'epochs': 8, 'warmup_ratio': 0.028, 'weight_decay': 0.0602, 'dropout': 0.158, 'attention_dropout': 0.075, 'gradient_accumulation_steps': 8, 'lr_scheduler_type': 'constant_with_warmup', 'label_smoothing': 0.031, 'max_grad_norm': 1.88, 'fitness': 0.4822291490531363}
2026-05-25 07:49:11,594 | tinylmtune._internal.trainer | Training: lr=0.0004805092517635441 bs=4 epochs=8 dropout=0.16 attn_drop=0.07 grad_accum=8 scheduler=constant_with_warmup label_smooth=0.031 grad_norm=1.9


Epoch,Training Loss,Validation Loss
0,5.992300,3.058789
1,2.275300,1.623100
2,1.206500,1.210114
3,0.797400,1.084173
4,0.622800,1.073703
5,0.552700,1.082936
6,0.521200,1.101153


There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


2026-05-25 07:49:32,765 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 1.0737029314041138, 'eval_runtime': 0.2409, 'eval_samples_per_second': 792.901, 'eval_steps_per_second': 199.263, 'epoch': 6.963350785340314}
2026-05-25 07:49:33,565 | tinylmtune._internal.inference | Model saved → /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/summarization_synthetic


Best config: {'learning_rate': 0.0004805092517635441, 'batch_size': 4, 'epochs': 8, 'warmup_ratio': 0.028, 'weight_decay': 0.0602, 'dropout': 0.158, 'attention_dropout': 0.075, 'gradient_accumulation_steps': 8, 'lr_scheduler_type': 'constant_with_warmup', 'label_smoothing': 0.031, 'max_grad_norm': 1.88, 'fitness': 0.4822291490531363, 'max_len': 96, 'task': 'summarization', 'output_dir': '/home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/summarization_synthetic', 'ga_history': [{'generation': 1, 'individual': 0, 'fitness': 0.3723381174514673, 'learning_rate': 0.0004524428304471338, 'batch_size': 16, 'epochs': 5, 'warmup_ratio': 0.239, 'weight_decay': 0.0153, 'dropout': 0.164, 'attention_dropout': 0.09, 'gradient_accumulation_steps': 1, 'lr_scheduler_type': 'constant_with_warmup', 'label_smoothing': 0.071, 'max_grad_norm': 2.02}, {'generation': 1, 'individual': 1, 'fitness': 0.4783739657509522, 'learning_rate': 0.0004805092517635441, 'batch_size': 4, 'epochs': 4, 'warmup_ratio': 0.

### Model Inference (Synthetic Data)

In [3]:
model = TinyInference("models/summarization_synthetic")
result = model.predict(
    "The European Space Agency announced today that its new Mars rover "
    "has successfully landed on the surface of Mars. The rover, named "
    "Athena, will spend the next two years collecting soil samples and "
    "analyzing the atmosphere for signs of past microbial life."
)
print("Output:", result)

2026-05-25 07:52:24,112 | tinylmtune._internal.inference | Loaded summarization model from models/summarization_synthetic
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Output: {'output': '. the european space company announced today that its new planets chopper has success transported on the sand of saturn. the chopper, called sandra, will spend the next two years analyzing soil fossils and analyzing the atmosphere for symbols of past smalleral life.'}


---
## Example 2 — Benchmark Data (xsum)

Uses a real HuggingFace dataset. No Flan-T5 needed.

### Load xsum dataset

In [4]:
from datasets import load_dataset

ds = load_dataset("xsum", split="train")
ds = ds.shuffle(seed=42).select(range(1000))

benchmark_data = [
    {"text": r["document"][:1000], "summary": r["summary"]}
    for r in ds
]

print(f"Loaded {len(benchmark_data)} records")
print(f"Text preview: {benchmark_data[0]['text'][:100]}...")
print(f"Summary: {benchmark_data[0]['summary']}")

Loaded 1000 records
Text preview: In Wales, councils are responsible for funding and overseeing schools.
But in England, Mr Osborne's ...
Summary: As Chancellor George Osborne announced all English state schools will become academies, the Welsh Government continues to reject the model here.


### Analyze token lengths

In [5]:
print_token_analysis(benchmark_data, task="summarization")

2026-05-25 07:53:15,235 | tinylmtune._internal.token_analyzer | Token analysis: n=999 min=35 mean=196 p95=240 max=441 → max_len=256, ga_choices=[256, 512]


  tinyLMTune — Token Length Analysis
  Task: summarization  |  Samples: 999

  Token lengths:  min=35  mean=196  median=212  max=441

  Percentiles:
    p 50:  212 tokens
    p 75:  224 tokens
    p 90:  233 tokens
    p 95:  240 tokens
    p100:  441 tokens

  Truncation at standard lengths:
    max_len=  2: 100.0% truncated  ██████████████████████████████████████████████████
    max_len=  4: 100.0% truncated  ██████████████████████████████████████████████████
    max_len=  8: 100.0% truncated  ██████████████████████████████████████████████████
    max_len= 16: 100.0% truncated  ██████████████████████████████████████████████████
    max_len= 24: 100.0% truncated  ██████████████████████████████████████████████████
    max_len= 32: 100.0% truncated  ██████████████████████████████████████████████████
    max_len= 64:  98.1% truncated  █████████████████████████████████████████████████
    max_len= 96:  91.7% truncated  █████████████████████████████████████████████
    max_len=128:  87.5% 

{'n_samples': 999,
 'min': 35,
 'max': 441,
 'mean': 196,
 'median': 212,
 'p50': 212,
 'p75': 224,
 'p90': 233,
 'p95': 240,
 'p100': 441,
 'truncation_pct': {2: 100.0,
  4: 100.0,
  8: 100.0,
  16: 100.0,
  24: 100.0,
  32: 100.0,
  64: 98.1,
  96: 91.7,
  128: 87.5,
  192: 77.2,
  256: 0.5,
  384: 0.2,
  512: 0.0},
 'recommended_max_len': 256,
 'recommended_ga_choices': [256, 512]}

### Check recommended search space

In [ ]:
print_recommendation(n_samples=len(benchmark_data), task="summarization")

### Train with GA optimisation

In [6]:
best = optimize_slm(
    task="summarization",
    user_data=benchmark_data,
    max_len=256,
    pop_size=4,
    generations=2,
    output_dir="models/summarization_benchmark",
)
print("Best config:", best)

2026-05-25 07:53:54,655 | tinylmtune._internal.pipeline | Model will be saved to: /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/summarization_benchmark
2026-05-25 07:53:54,656 | tinylmtune._internal.pipeline | user_data validated: 1000 / 1000 structured records
2026-05-25 07:53:54,656 | tinylmtune._internal.pipeline | Resolved 1000 structured records
2026-05-25 07:53:55,238 | tinylmtune._internal.token_analyzer | Token analysis: n=999 min=35 mean=196 p95=240 max=441 → max_len=256, ga_choices=[256, 512]
2026-05-25 07:53:55,242 | tinylmtune._internal.pipeline | Token analysis: p50=212 p95=240 max=441 → fixed max_len=256
2026-05-25 07:53:55,242 | tinylmtune._internal.dataset | Using 1000 user-provided records
2026-05-25 07:53:56,041 | tinylmtune._internal.dataset | Dataset: 800 train, 200 val
2026-05-25 07:53:56,041 | tinylmtune._internal.pipeline | Task=summarization, num_labels=2, n_train=800
2026-05-25 07:53:56,042 | tinylmtune._internal.pipeline | Fixed max_len=256 | GA sear

Epoch,Training Loss,Validation Loss
1,3.008600,0.778835
2,0.354900,0.680326
3,0.116000,0.723758
4,0.064300,0.714994


There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


2026-05-25 07:54:24,003 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 0.6803258657455444, 'eval_runtime': 0.4336, 'eval_samples_per_second': 461.273, 'eval_steps_per_second': 115.318, 'epoch': 4.0}
2026-05-25 07:54:24,218 | tinylmtune._internal.trainer | Training: lr=0.0001201671422284161 bs=4 epochs=10 dropout=0.11 attn_drop=0.04 grad_accum=4 scheduler=linear label_smooth=0.076 grad_norm=1.2


Epoch,Training Loss,Validation Loss
1,7.199100,4.261893
2,3.237900,2.356213
3,2.128400,1.897825
4,1.772500,1.738088
5,1.601300,1.662927
6,1.497500,1.621508
7,1.431700,1.598217
8,1.387100,1.583032
9,1.358600,1.578971
10,1.344400,1.576934


...

2026-05-25 07:58:27,878 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 0.6050084233283997, 'eval_runtime': 0.4323, 'eval_samples_per_second': 462.659, 'eval_steps_per_second': 115.665, 'epoch': 5.0}
2026-05-25 07:58:27,879 | tinylmtune._internal.ga_optimizer | Gen 2/2 — best=0.6230  avg=0.6002  worst=0.5859
2026-05-25 07:58:27,879 | tinylmtune._internal.pipeline | Best config (fitness=0.6230): {'learning_rate': 0.00018085899661262315, 'batch_size': 4, 'epochs': 6, 'warmup_ratio': 0.073, 'weight_decay': 0.0635, 'dropout': 0.02, 'attention_dropout': 0.092, 'gradient_accumulation_steps': 1, 'lr_scheduler_type': 'constant_with_warmup', 'label_smoothing': 0.003, 'max_grad_norm': 1.96, 'fitness': 0.6230496896248318}
2026-05-25 07:58:28,108 | tinylmtune._internal.trainer | Training: lr=0.00018085899661262315 bs=4 epochs=6 dropout=0.02 attn_drop=0.09 grad_accum=1 scheduler=constant_with_warmup label_smooth=0.003 grad_norm=2.0


Epoch,Training Loss,Validation Loss
1,3.556000,0.899183
2,0.503100,0.630483
3,0.208800,0.605008
4,0.115100,0.618854
5,0.086300,0.631424


There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


2026-05-25 07:59:03,033 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 0.6050084233283997, 'eval_runtime': 0.434, 'eval_samples_per_second': 460.851, 'eval_steps_per_second': 115.213, 'epoch': 5.0}
2026-05-25 07:59:03,195 | tinylmtune._internal.inference | Model saved → /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/summarization_benchmark


Best config: {'learning_rate': 0.00018085899661262315, 'batch_size': 4, 'epochs': 6, 'warmup_ratio': 0.073, 'weight_decay': 0.0635, 'dropout': 0.02, 'attention_dropout': 0.092, 'gradient_accumulation_steps': 1, 'lr_scheduler_type': 'constant_with_warmup', 'label_smoothing': 0.003, 'max_grad_norm': 1.96, 'fitness': 0.6230496896248318, 'max_len': 256, 'task': 'summarization', 'output_dir': '/home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/summarization_benchmark', 'ga_history': [{'generation': 1, 'individual': 0, 'fitness': 0.5951226606610079, 'learning_rate': 0.00032151626523665245, 'batch_size': 4, 'epochs': 6, 'warmup_ratio': 0.073, 'weight_decay': 0.014, 'dropout': 0.02, 'attention_dropout': 0.148, 'gradient_accumulation_steps': 1, 'lr_scheduler_type': 'constant_with_warmup', 'label_smoothing': 0.003, 'max_grad_norm': 0.92}, {'generation': 1, 'individual': 1, 'fitness': 0.38805809550653614, 'learning_rate': 0.0001201671422284161, 'batch_size': 4, 'epochs': 10, 'warmup_ratio': 

### Visualize GA Results

5 plots showing how the GA searched for the best hyperparameters:
1. **Fitness progress** — best/avg/worst per generation
2. **Parameter scatter** — each param vs fitness (best = red star)
3. **Scheduler comparison** — box plot by LR scheduler type
4. **Config evolution** — how the best config changed over generations
5. **Population heatmap** — all individuals in the last generation

In [ ]:
from tinylmtune import plot_results, print_best_config_table

# Print formatted best config
print_best_config_table(best)

# Generate all 5 plots
figs = plot_results(best, save_dir="plots/benchmark")

### Inference on benchmark model

In [ ]:
model = TinyInference("models/summarization_benchmark")
result = model.predict("Scientists have discovered a new species of deep-sea fish.")
print("Output:", result)

---
## Example 3 — Raw User Data

Three sub-examples showing different input formats:
- **3a.** Structured dicts (correct format)
- **3b.** Raw text strings (auto-labelled via Flan-T5)
- **3c.** Wrong-format dicts (auto-detected and converted)

### 3a. Structured dicts (used directly, no Flan-T5)

In [ ]:
my_data = [
    {"text": "The company reported record quarterly earnings of $5 billion, driven by strong growth in its cloud computing division. CEO Jane Smith attributed the success to increased enterprise adoption and new AI-powered services launched in Q3.", "summary": "Company posts record $5B quarterly earnings from cloud growth."},
    {"text": "A magnitude 6.2 earthquake struck off the coast of Japan early Monday morning, triggering tsunami warnings across the Pacific. Authorities evacuated coastal communities and emergency services are assessing the damage.", "summary": "6.2 earthquake near Japan triggers tsunami warnings and evacuations."},
    {"text": "Researchers at MIT have developed a new type of battery that can charge in under five minutes and last for over 1000 cycles. The technology uses a novel lithium-iron-phosphate chemistry.", "summary": "MIT creates fast-charging battery lasting 1000+ cycles."},
    {"text": "The World Health Organization declared the mpox outbreak a global health emergency, urging countries to increase vaccination efforts and improve surveillance systems.", "summary": "WHO declares mpox a global health emergency."},
    {"text": "SpaceX successfully launched its 50th Starlink mission of the year, deploying 60 satellites into low Earth orbit to expand global internet coverage.", "summary": "SpaceX completes 50th Starlink launch with 60 satellites."},
    {"text": "New York City announced plans to ban gas-powered vehicles from Manhattan by 2035, making it the first major US city to implement such a policy.", "summary": "NYC to ban gas vehicles in Manhattan by 2035."},
    {"text": "Apple unveiled its latest mixed reality headset at WWDC, featuring eye tracking, spatial audio, and a new operating system designed for immersive computing.", "summary": "Apple launches mixed reality headset with eye tracking."},
    {"text": "A study published in Nature found that urban green spaces reduce mental health issues by up to 30 percent among city residents who visit them weekly.", "summary": "Urban parks cut mental health issues by 30% with weekly visits."},
    {"text": "The Federal Reserve held interest rates steady at 5.25 percent, signaling a wait-and-see approach amid mixed economic indicators and cooling inflation.", "summary": "Fed holds rates at 5.25% amid mixed economic signals."},
    {"text": "Formula One announced Las Vegas as a permanent fixture on its calendar, signing a 10-year deal to host a night race on the famous Strip.", "summary": "F1 signs 10-year Las Vegas night race deal."},
]

best = optimize_slm(
    task="summarization",
    user_data=my_data,
    pop_size=4,
    generations=2,
    output_dir="models/summarization_user",
)

### 3b. Raw text strings (requires Flan-T5)

In [ ]:
# Raw text block — split into paragraphs, structured via Flan-T5
raw_text = """
The global semiconductor shortage continued to impact automotive production 
in the third quarter. Major manufacturers reported delays of up to six months 
for new vehicle deliveries.

Meanwhile, chip makers are investing heavily in new fabrication facilities. 
TSMC announced a $40 billion expansion in Arizona, while Intel is building 
two new plants in Ohio.

Industry analysts predict the shortage will ease by mid-2025 as new capacity 
comes online. However, demand for advanced chips in AI applications may 
offset the additional supply.
"""

best = optimize_slm(
    task="summarization",
    user_data=raw_text,
    split_strategy="paragraph",
    pop_size=4,
    generations=1,
    output_dir="models/summarization_raw",
)

### 3c. Wrong-format dicts (requires Flan-T5)

In [ ]:
# Dicts with non-standard keys
wrong_format = [
    {"article": "Global temperatures reached a new record high in July, with average readings 1.5 degrees above pre-industrial levels.", "brief": "July breaks global temperature records."},
    {"article": "The tech giant announced layoffs affecting 10,000 employees as part of a restructuring to focus on AI development.", "brief": "Tech company cuts 10K jobs to pivot to AI."},
]

# Pipeline detects these don't match {"text", "summary"}, extracts text, structures via Flan-T5
best = optimize_slm(
    task="summarization",
    user_data=wrong_format,
    pop_size=4,
    generations=1,
    output_dir="models/summarization_wrong",
)

### Visualize user data results

In [ ]:
# Plot results from structured data training (Example 3a)
from tinylmtune import plot_results, print_best_config_table
print_best_config_table(best)
figs = plot_results(best, save_dir="plots/user_data")

### Inference

In [ ]:
model = TinyInference("models/summarization_user")
result = model.predict("The central bank raised interest rates by 25 basis points.")
print(result)

---
## Summary

| Example | Data source | Flan-T5 needed | Best for |
|---------|-------------|---------------|----------|
| Synthetic | Auto-generated | Yes | Quick prototyping |
| Benchmark | xsum | No | Reproducible evaluation |
| User data | Your own text | Depends on format | Production use |

The GA searches 11 hyperparameters: `learning_rate`, `batch_size`, `epochs`, `warmup_ratio`, `weight_decay`, `dropout`, `attention_dropout`, `gradient_accumulation_steps`, `lr_scheduler_type`, `label_smoothing`, `max_grad_norm`.

`max_len` is automatically determined from your data's token length distribution (p95 percentile).